In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from obspy import UTCDateTime
from flovopy.processing.sam import RSAM

LBSSP_DATA = Path("/Volumes/tachyon/LBSSP_DATA")
SAM_DIR = LBSSP_DATA / "nodal_rsam"
OUT_DIR = LBSSP_DATA / "nodal_qc" / "noise_profiles"
OUT_DIR.mkdir(parents=True, exist_ok=True)

NETWORKS = ["T1", "T3"]
LOCATIONS = ["N1", "N2", "N3"]
CHANNELS = ["DPE", "DPN", "DPZ"]

BANDS = [
    "rms",
    "B4_8",
    "B8_16",
    "B16_32",
    "B32_64",
    "B64_128",
    "B128_240",
]

WINDOWS = [
    {"period": "overnight_2026-05-17_0400_0800", "group": "overnight",
     "start": UTCDateTime("2026-05-17T04:00:00"), "end": UTCDateTime("2026-05-17T08:00:00")},
    {"period": "overnight_2026-05-18_0400_0800", "group": "overnight",
     "start": UTCDateTime("2026-05-18T04:00:00"), "end": UTCDateTime("2026-05-18T08:00:00")},
    {"period": "overnight_2026-05-19_0400_0800", "group": "overnight",
     "start": UTCDateTime("2026-05-19T04:00:00"), "end": UTCDateTime("2026-05-19T08:00:00")},
    {"period": "may18_1100_1300_pre_site", "group": "pre_site",
     "start": UTCDateTime("2026-05-18T11:00:00"), "end": UTCDateTime("2026-05-18T13:00:00")},
]

START = min(w["start"] for w in WINDOWS)
END = max(w["end"] for w in WINDOWS)


def station_sort_key(seed_id):
    sta = seed_id.split(".")[1]
    try:
        return int(sta)
    except ValueError:
        return sta


def summarize_rsam(rsam):
    rows = []

    for seed_id, df in rsam.dataframes.items():
        parts = seed_id.split(".")
        if len(parts) != 4:
            continue

        net, sta, loc, chan = parts

        if loc not in LOCATIONS:
            continue

        if chan not in CHANNELS:
            continue

        for win in WINDOWS:
            mask = (
                (df["time"] >= win["start"].timestamp)
                & (df["time"] < win["end"].timestamp)
            )

            for band in BANDS:
                if band not in df.columns:
                    continue

                x = df.loc[mask, band].dropna()

                if x.empty:
                    continue

                rows.append({
                    "network": net,
                    "deployment": net,
                    "seed_id": seed_id,
                    "station": sta,
                    "location": loc,
                    "channel": chan,
                    "component": chan[-1],
                    "period": win["period"],
                    "group": win["group"],
                    "band": band,
                    "n": len(x),
                    "median": x.median(),
                    "mean": x.mean(),
                    "p25": x.quantile(0.25),
                    "p75": x.quantile(0.75),
                })

    summary = pd.DataFrame(rows)

    if summary.empty:
        return summary

    seed_order = sorted(summary["seed_id"].unique(), key=station_sort_key)
    seed_to_node = {seed_id: i + 1 for i, seed_id in enumerate(seed_order)}
    summary["node"] = summary["seed_id"].map(seed_to_node)

    return summary


all_summaries = []

for network in NETWORKS:
    print(f"\nReading RSAM for {network}")

    rsam = RSAM.read(
        START,
        END,
        SAM_DIR=str(SAM_DIR),
        network=network,
        sampling_interval=60,
        ext="csv",
        verbose=True,
    )

    summary = summarize_rsam(rsam)

    if summary.empty:
        print(f"No data summarized for {network}")
        continue

    all_summaries.append(summary)

if all_summaries:
    combined = pd.concat(all_summaries, ignore_index=True)

    outfile = OUT_DIR / "nodal_noise_profile_summary_all_components.csv"
    combined.to_csv(outfile, index=False)
    print(f"\nWrote {outfile}")

    print(combined.groupby(["network", "location", "component"])["station"].nunique())
else:
    print("No summaries generated.")


Reading RSAM for T1
Reading /Volumes/tachyon/LBSSP_DATA/nodal_rsam/RSAM/T1/RSAM_T1.05726.N1.DPE_2026_60s.csv
Dataframe with 780 rows is already on a regular 60 s grid based on column 'time'
Reading /Volumes/tachyon/LBSSP_DATA/nodal_rsam/RSAM/T1/RSAM_T1.05726.N1.DPN_2026_60s.csv
Dataframe with 780 rows is already on a regular 60 s grid based on column 'time'
Reading /Volumes/tachyon/LBSSP_DATA/nodal_rsam/RSAM/T1/RSAM_T1.05726.N1.DPZ_2026_60s.csv
Dataframe with 780 rows is already on a regular 60 s grid based on column 'time'
Reading /Volumes/tachyon/LBSSP_DATA/nodal_rsam/RSAM/T1/RSAM_T1.05726.N2.DPE_2026_60s.csv
Dataframe with 2400 rows is already on a regular 60 s grid based on column 'time'
Reading /Volumes/tachyon/LBSSP_DATA/nodal_rsam/RSAM/T1/RSAM_T1.05726.N2.DPN_2026_60s.csv
Dataframe with 2400 rows is already on a regular 60 s grid based on column 'time'
Reading /Volumes/tachyon/LBSSP_DATA/nodal_rsam/RSAM/T1/RSAM_T1.05726.N2.DPZ_2026_60s.csv
Dataframe with 2400 rows is already on